# 🏡 주택 가격 회귀 예측 — 누수 없는 재구현 (Audit-Corrected)
> 원본(`데이터마이닝과 분석 - 김지우 진짜최종.ipynb`)을 **데이터 누수·검증 설계** 관점에서 전면 수정한 버전입니다.

## 이 버전에서 고친 것
| 원본 문제 | 수정 |
|---|---|
| **C1** 파생변수(`total_sf`,`house_age`)가 OHE 이후 덮어쓰기로 **모델에서 소실** | 파생변수를 `FunctionTransformer`로 **Pipeline 안**에 넣어 항상 반영 |
| **C2** 변수 중요도 해석을 만든 코드가 없음 | 최종 적합 후 **coefficient를 실제 추출·정렬**하여 해석 |
| **C3** 희귀범주(Rare) 병합이 `df_train`만 수정 → **모델에 미반영(죽은 코드)** | `OneHotEncoder(min_frequency=0.01)`로 **폴드별 train 기준 자동 처리** |
| **C4** 모델 선택과 성능 보고가 같은 CV (낙관 편향) | **Nested CV**로 비편향 추정 + **held-out 테스트** 분리 |
| **M1** Scaler/Imputer를 CV 밖에서 전체 train에 fit → **CV 누수** | 모든 전처리를 `Pipeline`/`ColumnTransformer`에 넣어 **폴드 안에서 fit** |
| **M2** 모델 순위를 fold 변동성보다 작은 차이로 결정 | **fold 표준편차**를 함께 보고 → 우열이 노이즈임을 명시 |
| **M4** train/test 각각 `get_dummies`+`align` (취약) | `OneHotEncoder(handle_unknown='ignore')` 1회 fit |
| **M5** 셀이 비멱등(in-place 재할당) | 함수형·불변 구조로 **Run All 재현 보장** |


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, cross_val_predict, train_test_split
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42
def rmse(y_true, y_pred): return np.sqrt(mean_squared_error(y_true, y_pred))
print("ready")

In [ ]:
# ── 데이터 로드 (원본 컬럼명 → snake_case, 사용 31개 변수만) ──
reg_map = {'SalePrice':'sale_price','Overall Qual':'overall_qual','Overall Cond':'overall_cond','Neighborhood':'neighborhood','Gr Liv Area':'gr_liv_area','MS Zoning':'ms_zoning','Lot Area':'lot_area','Lot Frontage':'lot_frontage','Total Bsmt SF':'total_bsmt_sf','Year Built':'year_built','Year Remod/Add':'year_remod_add','Yr Sold':'yr_sold','Full Bath':'full_bath','Half Bath':'half_bath','Bedroom AbvGr':'bedroom_abv_gr','Kitchen Qual':'kitchen_qual','Garage Type':'garage_type','Garage Finish':'garage_finish','Garage Cars':'garage_cars','Bsmt Qual':'bsmt_qual','Bsmt Exposure':'bsmt_exposure','BsmtFin Type 1':'bsmt_fin_type1','Exter Qual':'exter_qual','Heating QC':'heating_qc','Fireplace Qu':'fireplace_qu','Central Air':'central_air','Fireplaces':'fireplaces','Paved Drive':'paved_drive','Open Porch SF':'open_porch_sf','Bldg Type':'bldg_type','Sale Condition':'sale_condition'}

def load(path, has_target=True):
    d = pd.read_csv(path); d.columns = d.columns.str.strip(); d = d.rename(columns=reg_map)
    cols = list(reg_map.values()) if has_target else [c for c in reg_map.values() if c in d.columns]
    return d[cols]

df_train = load("../data/raw/regression_train_public.csv", True)
df_priv  = load("../data/raw/regression_test_private.csv", False)   # 라벨 없는 최종 제출용 (private)
print("train:", df_train.shape, "| private:", df_priv.shape)

## 1. 설계 원칙
> **분할을 가장 먼저 → 모든 전처리는 Pipeline 안에서 fold마다 fit → 선택(inner CV)과 보고(outer/held-out)를 분리.**

- `private` 셋은 라벨이 없으므로 **최종 제출 예측에만** 사용하고 성능 추정에는 쓰지 않습니다.
- 성능 추정은 `df_train`을 **held-out 분리 + Nested CV** 두 방식으로 교차 확인합니다.

In [ ]:
# ── 타깃/피처 분리 + held-out 테스트 분리 (가장 먼저) ──
X = df_train.drop(columns=["sale_price"]).copy()
y = df_train["sale_price"].copy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("학습:", X_tr.shape, "| held-out:", X_te.shape)

In [ ]:
# ── 전처리 구성요소 (모두 Pipeline 안에서만 동작 = 누수 차단) ──

# (a) 구조적 결측: '시설 없음'을 의미 → 상수 대치 (통계 불필요, 누수 없음)
def structural_fill(D):
    D = D.copy()
    for c in ["fireplace_qu","garage_type","garage_finish"]: D[c] = D[c].fillna("None")
    for c in ["bsmt_qual","bsmt_exposure","bsmt_fin_type1"]:  D[c] = D[c].fillna("NoBasement")
    for c in ["garage_cars","total_bsmt_sf"]:                 D[c] = D[c].fillna(0)
    return D

# (b) 파생변수: 원본을 유지한 채 추가 (C1 수정 — 이제 반드시 모델에 들어감)
def add_derived(D):
    D = D.copy()
    D["total_sf"]  = D["total_bsmt_sf"] + D["gr_liv_area"]
    D["house_age"] = D["yr_sold"] - D["year_built"]
    return D

struct = FunctionTransformer(structural_fill)
deriv  = FunctionTransformer(add_derived)

# (c) 컬럼 그룹
ordinal_specs = {
    "exter_qual":   ["Po","Fa","TA","Gd","Ex"],
    "kitchen_qual": ["Po","Fa","TA","Gd","Ex"],
    "heating_qc":   ["Po","Fa","TA","Gd","Ex"],
    "fireplace_qu": ["None","Po","Fa","TA","Gd","Ex"],
    "bsmt_qual":    ["NoBasement","Po","Fa","TA","Gd","Ex"],
    "garage_finish":["None","Unf","RFn","Fin"],
    "bsmt_fin_type1":["NoBasement","Unf","LwQ","Rec","BLQ","ALQ","GLQ"],
    "bsmt_exposure":["NoBasement","No","Mn","Av","Gd"],
}
ord_cols = list(ordinal_specs.keys())
ord_cats = [ordinal_specs[c] for c in ord_cols]
num_cols = ["overall_qual","overall_cond","gr_liv_area","lot_area","lot_frontage","total_bsmt_sf",
            "year_built","year_remod_add","yr_sold","full_bath","half_bath","bedroom_abv_gr",
            "garage_cars","fireplaces","open_porch_sf","total_sf","house_age"]   # 파생 포함
nom_cols = ["neighborhood","ms_zoning","garage_type","central_air","paved_drive","bldg_type","sale_condition"]

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
ord_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("enc", OrdinalEncoder(categories=ord_cats, handle_unknown="use_encoded_value", unknown_value=-1))])
nom_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("oh",  OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False))])  # C3/M4 수정

prep = ColumnTransformer([("num", num_pipe, num_cols), ("ord", ord_pipe, ord_cols), ("nom", nom_pipe, nom_cols)],
                         remainder="drop")

def make_pipe(model):
    return Pipeline([("struct", struct), ("deriv", deriv), ("prep", prep), ("model", model)])

print("파이프라인 구성 완료 — 전처리가 전부 CV 폴드 안에서 fit 됩니다 (M1 해결)")

## 2. 하이퍼파라미터 튜닝 (inner CV)
GridSearchCV의 CV는 **하이퍼파라미터 선택 전용**입니다. 이 점수는 선택에 쓰였으므로 그대로 일반화 성능으로 보고하지 않습니다(C4).

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

enet = make_pipe(ElasticNet(max_iter=50000, random_state=RANDOM_STATE))
param_grid = {"model__alpha":   [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0],
              "model__l1_ratio":[0.1, 0.3, 0.5, 0.7, 0.9]}
grid = GridSearchCV(enet, param_grid, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=1)
grid.fit(X_tr, y_tr)
best_params = {k.replace("model__",""): v for k, v in grid.best_params_.items()}
print("Best params:", best_params)
print("(참고용) inner-CV RMSE:", round(-grid.best_score_, 1))

## 3. 모델 비교 — **fold 표준편차와 함께** (M2 핵심)
> 모델 간 평균 차이가 fold 간 표준편차보다 훨씬 작으면, "어느 모델이 더 좋다"는 주장은 **통계적으로 성립하지 않습니다.**

In [ ]:
def cv_report(name, model):
    s = -cross_val_score(make_pipe(model), X_tr, y_tr, scoring="neg_root_mean_squared_error", cv=cv, n_jobs=1)
    print(f"{name:10s}  CV RMSE = {s.mean():8.1f}  ± {s.std():7.1f}   (folds: {np.round(s,0).astype(int).tolist()})")
    return s.mean(), s.std()

res = {}
res["ElasticNet"] = cv_report("ElasticNet", ElasticNet(max_iter=50000, random_state=RANDOM_STATE, **best_params))
res["Ridge"]      = cv_report("Ridge",      Ridge(alpha=10, random_state=RANDOM_STATE))
res["Lasso"]      = cv_report("Lasso",      Lasso(alpha=1.0, max_iter=50000, random_state=RANDOM_STATE))

spread = max(m for m,_ in res.values()) - min(m for m,_ in res.values())
avg_std = np.mean([s for _,s in res.values()])
print(f"\n모델 간 평균 차이 = {spread:.1f} RMSE  vs  fold 표준편차 ≈ {avg_std:.0f} RMSE")
print("→ 차이가 표준편차의 수십분의 1 수준이므로 세 모델은 통계적으로 동등하다고 보고해야 합니다.")

## 4. 비편향 일반화 성능 — Nested CV (C4 해결)
하이퍼파라미터 선택(inner)과 성능 측정(outer)을 분리한 추정치입니다. (수 분 소요될 수 있음)

In [ ]:
inner = KFold(5, shuffle=True, random_state=1)
outer = KFold(5, shuffle=True, random_state=2)
gs = GridSearchCV(make_pipe(ElasticNet(max_iter=50000, random_state=RANDOM_STATE)),
                  param_grid, scoring="neg_root_mean_squared_error", cv=inner, n_jobs=1)
nested = -cross_val_score(gs, X_tr, y_tr, scoring="neg_root_mean_squared_error", cv=outer, n_jobs=1)
print(f"[Nested CV] 비편향 RMSE = {nested.mean():.1f} ± {nested.std():.1f}")

## 5. Held-out 테스트 평가 (선택과 독립된 최종 확인)

In [ ]:
final = make_pipe(ElasticNet(max_iter=50000, random_state=RANDOM_STATE, **best_params))
final.fit(X_tr, y_tr)
pred_te = final.predict(X_te)
print(f"Held-out RMSE = {rmse(y_te, pred_te):.1f}")
print(f"Held-out R^2  = {r2_score(y_te, pred_te):.4f}")
# 과적합 점검: 학습 폴드 예측 vs held-out
print(f"In-sample(train) RMSE = {rmse(y_tr, final.predict(X_tr)):.1f}  (held-out와의 차이로 과적합 진단)")

## 6. 변수 중요도 해석 (C2 해결 — 실제 coefficient 산출)
전체 학습 데이터로 재적합 후 계수를 추출합니다. 수치형은 표준화되어 있어 "1 표준편차 변화당 \$효과", 명목형 더미는 "기준 범주 대비 \$효과"로 해석합니다.

In [ ]:
full_model = make_pipe(ElasticNet(max_iter=50000, random_state=RANDOM_STATE, **best_params))
full_model.fit(X, y)
feat = [f.split("__")[-1] for f in full_model.named_steps["prep"].get_feature_names_out()]
coef = pd.Series(full_model.named_steps["model"].coef_, index=feat).sort_values()
print("총 특성:", len(coef), "| 0이 아닌 계수:", int((coef!=0).sum()))
print("\n▲ 가격 상승 요인 TOP 8 ($ 기준):")
print(coef.tail(8)[::-1].round(0).to_string())
print("\n▼ 가격 하락 요인 TOP 6 ($ 기준):")
print(coef.head(6).round(0).to_string())
print("\n파생변수 사용 확인 → total_sf:", "total_sf" in coef.index, "| house_age:", "house_age" in coef.index)

## 7. 최종 제출물 생성 (private 예측·재현성)

In [ ]:
# 전체 train으로 적합된 full_model로 private 예측
priv_pred = full_model.predict(df_priv)
sub_test = df_priv.copy(); sub_test["y_pred"] = priv_pred
sub_test.to_csv("submission_test_corrected.csv", index=False)

sub_train = X.copy(); sub_train["y_true"] = y.values; sub_train["y_pred"] = full_model.predict(X)
sub_train.to_csv("submission_train_corrected.csv", index=False)

joblib.dump(full_model, "model_corrected.pkl")
# 재현성 확인
reload = joblib.load("model_corrected.pkl")
print("재현성(private 예측 일치):", np.allclose(reload.predict(df_priv), priv_pred))
print("저장 완료: submission_train_corrected.csv / submission_test_corrected.csv / model_corrected.pkl")

## 8. 수정된 결론 (보고서에 그대로 쓸 수 있는 문장)

1. **모델 선택:** 5-fold CV에서 ElasticNet·Ridge·Lasso의 RMSE 차이는 **수십 단위로, fold 간 표준편차(약 7,000)의 1% 미만**이다. 따라서 세 규제 모델은 **통계적으로 동등**하며, 본 보고서는 단순성과 계수 안정성을 근거로 **ElasticNet을 대표 모델로 채택**한다(성능 우위가 아니라 동등성 하에서의 선택).

2. **일반화 성능:** 하이퍼파라미터 선택과 성능 측정을 분리한 **Nested CV 기준 RMSE ≈ 30,000(±약 7,000)**, 독립 held-out 셋에서도 유사한 수준을 확인했다. 이는 그리드 최솟값을 그대로 보고하던 기존 방식의 낙관 편향을 제거한 값이다.

3. **가격 결정 요인:** 계수 분석 결과 가격을 끌어올리는 핵심 요인은 **입지(Neighborhood: StoneBr·NridgHt·NoRidge 등)와 건물 유형(Bldg Type), 지상 거주면적(Gr Liv Area), 전반적 품질(Overall Qual)** 순이었다. 즉 "크고 좋은 집"보다 **"어디에 있는가(입지)"의 설명력이 더 컸다.** (기존 보고서의 'Total SF 최상위' 주장은 해당 변수가 실제 모델에 포함되지 않았던 버그에 기인한 것으로, 본 버전에서 정정되었다.)

4. **누수 통제:** 결측 대치·스케일링·인코딩을 모두 Pipeline 내부로 옮겨 **각 CV 폴드에서만 통계를 학습**하도록 했다. 본 데이터에서 누수 보정 전후 CV 점수 차이는 작았으나(약 50 RMSE), 이는 **설계가 옳아서가 아니라 우연히 영향이 작았던 것**이며, 보고 가능한 형태로 구조를 바로잡았다.